In [1]:
!pip install neuroCombat nibabel nilearn networkx seaborn xgboost -q

import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.fft import fft, fftfreq
from statsmodels.stats.multitest import multipletests
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix, roc_curve, auc
from xgboost import XGBClassifier
from neuroCombat import neuroCombat
from scipy.sparse.csgraph import shortest_path as sp_shortest_path
from sklearn.cross_decomposition import PLSRegression
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

os.makedirs('/kaggle/working/results', exist_ok=True)
os.makedirs('/kaggle/working/plots', exist_ok=True)

DATA_DIR = '/kaggle/input/datasets/saeedrezaeiafshar/gsr-abide-dataset/abide_global'
ROIS_DIR = f'{DATA_DIR}/rois_cc200'
PHENO_URL = 'https://s3.amazonaws.com/fcp-indi/data/Projects/ABIDE_Initiative/Phenotypic_V1_0b_preprocessed1.csv'

NETWORKS = {1: "Visual", 2: "SomatoMotor", 3: "DorsalAttn", 4: "Salience/VentAttn", 5: "Limbic", 6: "Frontoparietal", 7: "Default"}

ROI_TO_NET = {}
for i in range(1, 31): ROI_TO_NET[i] = 1
for i in range(31, 56): ROI_TO_NET[i] = 2
for i in range(56, 81): ROI_TO_NET[i] = 3
for i in range(81, 106): ROI_TO_NET[i] = 4
for i in range(106, 126): ROI_TO_NET[i] = 5
for i in range(126, 161): ROI_TO_NET[i] = 6
for i in range(161, 201): ROI_TO_NET[i] = 7

pheno_df = pd.read_csv(PHENO_URL)
file_to_sub = dict(zip(pheno_df['FILE_ID'], pheno_df['SUB_ID']))
sub_to_site = dict(zip(pheno_df['SUB_ID'], pheno_df['SITE_ID']))
sub_to_dx = dict(zip(pheno_df['SUB_ID'], pheno_df['DX_GROUP']))
sub_to_iq = dict(zip(pheno_df['SUB_ID'], pheno_df['FIQ']))

roi_files = glob.glob(f'{ROIS_DIR}/*.1D')
print(f"Found {len(roi_files)} ROI files")

def extract_file_id(filename):
    base = os.path.basename(filename)
    return base.replace('_rois_cc200.1D', '')

def load_ts(path):
    return np.loadtxt(path)

def compute_fc(ts):
    corr = np.corrcoef(ts.T)
    corr = np.clip(corr, -0.9999, 0.9999)
    fc = np.arctanh(corr)
    np.fill_diagonal(fc, 0)
    return fc

def compute_falff(ts, tr=2.0):
    n_t, n_r = ts.shape
    freqs = fftfreq(n_t, tr)
    low = (np.abs(freqs) >= 0.01) & (np.abs(freqs) <= 0.08)
    falff = np.zeros(n_r)
    for i in range(n_r):
        spec = np.abs(fft(ts[:, i]))**2
        falff[i] = np.sum(spec[low]) / np.sum(spec) if np.sum(spec) > 0 else 0
    return falff

def binarize(falff):
    thresh = np.median(falff)
    return np.where(falff >= thresh, 1, -1)

def hamiltonian(fc, signs):
    n = len(signs)
    e = 0.0
    for i in range(n):
        for j in range(i+1, n):
            e += signs[i] * fc[i,j] * signs[j]
    return -e

def edge_types(fc, signs):
    n = len(signs)
    counts = {'agreement': 0, 'disagreement': 0, 'imbalanced_same': 0, 'imbalanced_opp': 0}
    for i in range(n):
        for j in range(i+1, n):
            w = fc[i,j]
            if np.isnan(w): continue
            ls = 1 if w > 0 else -1
            same = signs[i] == signs[j]
            if same and ls == 1: counts['agreement'] += 1
            elif not same and ls == -1: counts['disagreement'] += 1
            elif same and ls == -1: counts['imbalanced_same'] += 1
            else: counts['imbalanced_opp'] += 1
    total = sum(counts.values()) or 1
    return {k: v/total for k, v in counts.items()}

def bipolarity(fc):
    S = np.sign(fc)
    np.fill_diagonal(S, 0)
    D = np.diag(np.abs(S).sum(axis=1))
    L = D - S
    try:
        vals, vecs = np.linalg.eigh(L)
        fiedler = vecs[:, 1]
        part = np.sign(fiedler)
        n = len(fc)
        bal, tot = 0, 0
        for i in range(n):
            for j in range(i+1, n):
                if np.isnan(fc[i,j]): continue
                s = 1 if fc[i,j] > 0 else -1
                bal += int(s * part[i] * part[j] > 0)
                tot += 1
        return bal / tot if tot > 0 else np.nan
    except:
        return np.nan

# ── Graph efficiency (addresses Comment 2: additional network metrics) ──────
def compute_efficiency(fc):
    """
    Compute global and local efficiency on the absolute-value weighted FC matrix.
    Uses the standard formulation: d_ij = 1 / |w_ij|, then
    global_eff = mean(1 / d_ij) across all i≠j.
    Local efficiency = mean over nodes of the efficiency of each node's
    immediate neighbourhood subgraph.
    """
    n = fc.shape[0]
    W = np.abs(fc.copy())
    np.fill_diagonal(W, 0)

    # Distance matrix: d = 1/w (0-weight → infinite distance)
    with np.errstate(divide='ignore', invalid='ignore'):
        D = np.where(W > 0, 1.0 / W, 0.0)

    # Global efficiency via Floyd-Warshall (scipy)
    D_inf = np.where(D > 0, D, np.inf)
    np.fill_diagonal(D_inf, 0)
    dist = sp_shortest_path(D_inf, method='D', directed=False)
    with np.errstate(divide='ignore', invalid='ignore'):
        inv_dist = np.where(dist > 0, 1.0 / dist, 0.0)
    global_eff = inv_dist.sum() / (n * (n - 1))

    # Local efficiency: efficiency of each node's neighbourhood
    local_effs = []
    for i in range(n):
        nbrs = np.where(W[i] > 0)[0]
        k = len(nbrs)
        if k < 2:
            local_effs.append(0.0)
            continue
        sub_D = np.where(W[np.ix_(nbrs, nbrs)] > 0,
                         1.0 / W[np.ix_(nbrs, nbrs)], np.inf)
        np.fill_diagonal(sub_D, 0)
        sub_dist = sp_shortest_path(sub_D, method='D', directed=False)
        with np.errstate(divide='ignore', invalid='ignore'):
            sub_inv = np.where(sub_dist > 0, 1.0 / sub_dist, 0.0)
        local_effs.append(sub_inv.sum() / (k * (k - 1)))
    local_eff = np.mean(local_effs)

    return global_eff, local_eff

def extract_features(fc, signs, roi_to_net):
    n_rois = len(signs)
    feats = {}
    feats['Whole_Energy_global'] = hamiltonian(fc, signs)
    feats['Whole_Bipolarity'] = bipolarity(fc)
    # Graph efficiency (requested by reviewer)
    g_eff, l_eff = compute_efficiency(fc)
    feats['Whole_GlobalEfficiency'] = g_eff
    feats['Whole_LocalEfficiency']  = l_eff
    
    props = edge_types(fc, signs)
    for k, v in props.items():
        feats[f'Whole_Prop_{k}'] = v
    net_ids = list(NETWORKS.keys())
    n_nets = len(net_ids)
    SumW = np.zeros((n_nets, n_nets))
    CntW = np.zeros((n_nets, n_nets))
    SumF = np.zeros((n_nets, n_nets))
    CntF = np.zeros((n_nets, n_nets))
    Eblk = np.zeros((n_nets, n_nets))
    for i in range(n_rois):
        for j in range(i+1, n_rois):
            ni, nj = roi_to_net.get(i+1), roi_to_net.get(j+1)
            if ni is None or nj is None: continue
            w = fc[i, j]
            if np.isnan(w): continue
            ii, jj = net_ids.index(ni), net_ids.index(nj)
            si, sj = signs[i], signs[j]
            SumW[ii, jj] += w; CntW[ii, jj] += 1
            SumF[ii, jj] += w * si * sj; CntF[ii, jj] += 1
            Eblk[ii, jj] -= w * si * sj
            if ii != jj:
                SumW[jj, ii] += w; CntW[jj, ii] += 1
                SumF[jj, ii] += w * si * sj; CntF[jj, ii] += 1
                Eblk[jj, ii] -= w * si * sj
    with np.errstate(divide='ignore', invalid='ignore'):
        AvgW = np.where(CntW > 0, SumW / CntW, np.nan)
        AvgF = np.where(CntF > 0, SumF / CntF, np.nan)
    for idx, nid in enumerate(net_ids):
        name = NETWORKS[nid]
        feats[f'Intra_{name}_AvgWeight'] = AvgW[idx, idx]
        feats[f'Intra_{name}_AvgFalffW'] = AvgF[idx, idx]
        feats[f'Intra_{name}_EnergyBlock'] = Eblk[idx, idx]
    for i, ni in enumerate(net_ids):
        for j in range(i+1, len(net_ids)):
            nj = net_ids[j]
            pair = f'Inter_{NETWORKS[ni]}__{NETWORKS[nj]}'
            feats[f'{pair}_AvgWeight'] = AvgW[i, j]
            feats[f'{pair}_AvgFalffW'] = AvgF[i, j]
            feats[f'{pair}_EnergyBlock'] = Eblk[i, j]
    return feats

rows = []
for fpath in roi_files:
    file_id = extract_file_id(fpath)
    sub_id = file_to_sub.get(file_id)
    if sub_id is None: continue
    try:
        ts = load_ts(fpath)
        if ts.shape[0] < 50: continue
        if ts.ndim == 1 or ts.shape[1] != 200: continue
        fc = compute_fc(ts)
        falff = compute_falff(ts)
        signs = binarize(falff)
        feats = extract_features(fc, signs, ROI_TO_NET)
        feats['Subject'] = sub_id
        feats['Group'] = 'ASD' if sub_to_dx.get(sub_id) == 1 else 'Control'
        feats['Center'] = sub_to_site.get(sub_id, 'Unknown')
        iq_val = sub_to_iq.get(sub_id, np.nan)
        feats['IQ'] = np.nan if iq_val == -9999 else iq_val
        rows.append(feats)
    except:
        pass

features_df = pd.DataFrame(rows)
print(f"Extracted features: {features_df.shape}")
print(f"ASD: {sum(features_df['Group']=='ASD')}, Control: {sum(features_df['Group']=='Control')}")

def remove_outliers(df, col, factor=1.5):
    vals = df[col].dropna()
    q1, q3 = vals.quantile([0.25, 0.75])
    iqr = q3 - q1
    lo, hi = q1 - factor * iqr, q3 + factor * iqr
    return df[(df[col] >= lo) & (df[col] <= hi)].copy()

features_df = remove_outliers(features_df, 'Whole_Energy_global')
features_df = features_df.reset_index(drop=True)
print(f"After outlier removal: {len(features_df)}")
print(f"ASD: {sum(features_df['Group']=='ASD')}, Control: {sum(features_df['Group']=='Control')}")

features_df.to_csv('/kaggle/working/results/features_raw_GSR.csv', index=False)
print("Saved raw features")

meta_cols = ['Subject', 'Group', 'Center', 'IQ']
feat_cols = [c for c in features_df.columns if c not in meta_cols]
print(f"Feature columns to harmonize: {len(feat_cols)}")

site_encoder = {site: i for i, site in enumerate(features_df['Center'].unique())}
covars = pd.DataFrame({
    'SITE': features_df['Center'].map(site_encoder).values,
    'Group': features_df['Group'].map({'ASD': 1, 'Control': 0}).values,
    'IQ': features_df['IQ'].fillna(features_df['IQ'].median()).values
})
print(f"Sites: {len(site_encoder)}")
print(f"Site mapping: {site_encoder}")

data_matrix = features_df[feat_cols].values.astype(np.float64)
print(f"Data matrix shape (subjects x features): {data_matrix.shape}")
print(f"NaN count before filling: {np.isnan(data_matrix).sum()}")

for j in range(data_matrix.shape[1]):
    col = data_matrix[:, j]
    col_median = np.nanmedian(col)
    col[np.isnan(col)] = col_median
    data_matrix[:, j] = col

print(f"NaN count after filling: {np.isnan(data_matrix).sum()}")
print(f"Data range: [{data_matrix.min():.3f}, {data_matrix.max():.3f}]")

data_for_combat = data_matrix.T
print(f"Data for ComBat shape (features x subjects): {data_for_combat.shape}")

print("\nRunning ComBat harmonization...")
try:
    combat_result = neuroCombat(dat=data_for_combat, covars=covars, batch_col='SITE')
    harmonized_data = combat_result['data']
    print(f"ComBat output shape: {harmonized_data.shape}")
    print(f"ComBat output NaN count: {np.isnan(harmonized_data).sum()}")
    print(f"ComBat output range: [{np.nanmin(harmonized_data):.3f}, {np.nanmax(harmonized_data):.3f}]")
    
    if np.isnan(harmonized_data).sum() > 0.5 * harmonized_data.size:
        print("WARNING: Too many NaNs in ComBat output, using raw data")
        harmonized_data = data_for_combat
    
    harmonized_df = features_df[meta_cols].copy()
    for i, col in enumerate(feat_cols):
        harmonized_df[col] = harmonized_data[i, :]
    
except Exception as e:
    print(f"ComBat failed with error: {e}")
    print("Using raw features instead")
    harmonized_df = features_df.copy()

harmonized_df['FIQ'] = harmonized_df['IQ']
harmonized_df.to_csv('/kaggle/working/results/features_ComBat_GSR.csv', index=False)
print("Saved harmonized features")

print(f"\nVerification:")
print(f"  Shape: {harmonized_df.shape}")
print(f"  Whole_Energy_global sample: {harmonized_df['Whole_Energy_global'].head(3).tolist()}")
print(f"  Whole_Energy_global NaN: {harmonized_df['Whole_Energy_global'].isna().sum()}")

# ── Comment 1: document normality tests explicitly ───────────────────────────
def adaptive_test_full(g1, g2, alpha=0.05):
    """
    Returns (test_name, stat, p, sw_stat_g1, sw_p_g1, sw_stat_g2, sw_p_g2,
             levene_stat, levene_p).
    Decision rule: if BOTH groups pass Shapiro-Wilk (p > alpha) → t-test,
    otherwise → Mann-Whitney U.  Levene is reported for completeness.
    """
    if len(g1) < 3 or len(g2) < 3:
        return ('skip',) + (np.nan,) * 8

    sw1, sp1 = stats.shapiro(g1)
    sw2, sp2 = stats.shapiro(g2)
    lev_stat, lev_p = stats.levene(g1, g2)

    if sp1 > alpha and sp2 > alpha:
        stat, p = stats.ttest_ind(g1, g2, equal_var=(lev_p > alpha))
        test_name = 't-test (Welch)' if lev_p <= alpha else 't-test'
    else:
        stat, p = stats.mannwhitneyu(g1, g2, alternative='two-sided')
        test_name = 'Mann-Whitney U'

    return test_name, stat, p, sw1, sp1, sw2, sp2, lev_stat, lev_p


asd  = harmonized_df[harmonized_df['Group'] == 'ASD']
ctrl = harmonized_df[harmonized_df['Group'] == 'Control']
print(f"\nStats: ASD={len(asd)}, Control={len(ctrl)}")

meta  = {'Subject', 'Group', 'Center', 'IQ', 'FIQ'}
fcols = [c for c in harmonized_df.columns if c not in meta]
print(f"Feature columns for stats: {len(fcols)}")

categories = {'Whole': [], 'Intra': [], 'Inter': []}
for col in fcols:
    if col.startswith('Whole_'):   categories['Whole'].append(col)
    elif col.startswith('Intra_'): categories['Intra'].append(col)
    elif col.startswith('Inter_'): categories['Inter'].append(col)

all_results    = []
normality_rows = []   # ← saved separately to justify test choice

for cat, feats in categories.items():
    results = []
    for feat in feats:
        av = asd[feat].dropna().values
        cv = ctrl[feat].dropna().values
        if len(av) < 3 or len(cv) < 3:
            continue
        (test, stat, p,
         sw1, sp1, sw2, sp2,
         lev_stat, lev_p) = adaptive_test_full(av, cv)
        if test == 'skip':
            continue

        # Store normality info for the supplementary table
        normality_rows.append({
            'Feature': feat, 'Category': cat,
            'SW_stat_ASD': sw1, 'SW_p_ASD': sp1,
            'SW_stat_TD':  sw2, 'SW_p_TD':  sp2,
            'Levene_stat': lev_stat, 'Levene_p': lev_p,
            'ASD_normal': sp1 > 0.05, 'TD_normal': sp2 > 0.05,
            'Test_chosen': test
        })

        results.append({
            'Feature': feat, 'Category': cat,
            'ASD_Mean': av.mean(), 'Control_Mean': cv.mean(),
            'ASD_Std':  av.std(),  'Control_Std':  cv.std(),
            'Test_Used': test, 'p_value': p
        })

    print(f"{cat}: {len(results)} comparisons")
    if results:
        cat_df = pd.DataFrame(results)
        _, p_corr, _, _ = multipletests(cat_df['p_value'], method='fdr_bh')
        cat_df['p_corrected']         = p_corr
        cat_df['Significant_FDR']     = p_corr < 0.05
        cat_df['Significant_Uncorrected'] = cat_df['p_value'] < 0.05
        all_results.append(cat_df)

# Save normality justification table
normality_df = pd.DataFrame(normality_rows)
normality_df.to_csv('/kaggle/working/results/normality_tests_GSR.csv', index=False)
print(f"Saved normality tests: {len(normality_df)} features")

# Summary: how many features were non-normal in each group
n_nonnormal_asd = (~normality_df['ASD_normal']).sum()
n_nonnormal_td  = (~normality_df['TD_normal']).sum()
print(f"Features failing normality: ASD={n_nonnormal_asd}, TD={n_nonnormal_td} "
      f"(out of {len(normality_df)})")

if all_results:
    stats_results = (pd.concat(all_results, ignore_index=True)
                     .sort_values('p_value'))
    stats_results.to_csv('/kaggle/working/results/statistical_comparison_GSR.csv',
                         index=False)
    print(f"\nTop results:")
    print(stats_results.head(10)[
        ['Feature', 'ASD_Mean', 'Control_Mean',
         'p_value', 'p_corrected', 'Significant_FDR']
    ].to_string())
else:
    stats_results = pd.DataFrame()
# ═══════════════════════════════════════════════════════════════════════════
# BEHAVIORAL CORRELATION ANALYSIS  (Comment 2)
# Addresses: (a) sign/direction reporting, (b) PLS multivariate analysis,
#            (c) Hamiltonian vs other network metrics
# ═══════════════════════════════════════════════════════════════════════════

# ── Load behavioral scores from phenotypic CSV ───────────────────────────
BEHAV_COLS = {
    'ADI_R_SOCIAL_TOTAL_A':   'ADI-R Social',
    'ADI_R_VERBAL_TOTAL_BV':  'ADI-R Verbal',
    'ADI_R_ONSET_TOTAL_D':    'ADI-R Onset',
    'ADOS_TOTAL':             'ADOS Total',
    'ADOS_COMM':              'ADOS Communication',
}

# Merge behavioral scores into harmonized_df (ASD only)
behav_df = pheno_df[['SUB_ID'] + list(BEHAV_COLS.keys())].copy()
behav_df = behav_df.rename(columns=BEHAV_COLS)
behav_df['Subject'] = behav_df['SUB_ID']

# Replace ABIDE sentinel value -9999 with NaN
for col in BEHAV_COLS.values():
    behav_df[col] = pd.to_numeric(behav_df[col], errors='coerce')
    behav_df.loc[behav_df[col] < 0, col] = np.nan

asd_merged = (harmonized_df[harmonized_df['Group'] == 'ASD']
              .merge(behav_df[['Subject'] + list(BEHAV_COLS.values())],
                     on='Subject', how='left'))
print(f"\nASD with behavioral data: {len(asd_merged)}")
for col in BEHAV_COLS.values():
    n = asd_merged[col].notna().sum()
    print(f"  {col}: n={n}")

# ── (a) Pairwise Pearson + Spearman with r/ρ AND direction ───────────────
behav_score_cols = list(BEHAV_COLS.values())
network_feat_cols = [c for c in fcols if c not in meta]

corr_rows = []
for bscore in behav_score_cols:
    sub = asd_merged[['Subject', bscore] + network_feat_cols].dropna(
        subset=[bscore])
    n_sub = len(sub)
    if n_sub < 10:
        continue
    for feat in network_feat_cols:
        pair = sub[[feat, bscore]].dropna()
        n_pair = len(pair)
        if n_pair < 10:
            continue
        x = pair[feat].values
        y = pair[bscore].values
        pr, pp = stats.pearsonr(x, y)
        sr, sp = stats.spearmanr(x, y)
        corr_rows.append({
            'Network_Metric': feat,
            'Behavioral_Score': bscore,
            'n': n_pair,
            'Pearson_r': pr,   'Pearson_p': pp,
            'Spearman_rho': sr, 'Spearman_p': sp,
            # Flag direction explicitly for reviewer
            'Direction_Pearson': 'positive' if pr > 0 else 'negative',
            'Nominal_sig_Pearson':  pp < 0.05,
            'Nominal_sig_Spearman': sp < 0.05,
        })

corr_df = pd.DataFrame(corr_rows)
if not corr_df.empty:
    # FDR correction across all tests
    _, p_fdr_p, _, _ = multipletests(corr_df['Pearson_p'],  method='fdr_bh')
    _, p_fdr_s, _, _ = multipletests(corr_df['Spearman_p'], method='fdr_bh')
    corr_df['Pearson_p_FDR']  = p_fdr_p
    corr_df['Spearman_p_FDR'] = p_fdr_s
    corr_df['FDR_sig_Pearson']  = p_fdr_p < 0.05
    corr_df['FDR_sig_Spearman'] = p_fdr_s < 0.05
    corr_df = corr_df.sort_values('Pearson_p')
    corr_df.to_csv('/kaggle/working/results/behavioral_correlations_GSR.csv',
                   index=False)
    print(f"\nBehavioral correlations saved: {len(corr_df)} pairs")
    print("Nominal significant (Pearson p<0.05):",
          corr_df['Nominal_sig_Pearson'].sum())
    print("FDR significant (Pearson):",
          corr_df['FDR_sig_Pearson'].sum())
    # Print direction of Whole_Energy_global correlations explicitly
    energy_corrs = corr_df[corr_df['Network_Metric'] == 'Whole_Energy_global']
    print("\n── Whole_Energy_global correlations (direction clarification) ──")
    print(energy_corrs[['Behavioral_Score', 'Pearson_r', 'Pearson_p',
                         'Spearman_rho', 'Spearman_p',
                         'Direction_Pearson']].to_string(index=False))

# ── (b) PLS brain-behavior multivariate analysis ─────────────────────────
print("\n── PLS analysis ──")

pls_results = []
for bscore in behav_score_cols:
    sub = asd_merged[['Subject', bscore] + network_feat_cols].dropna(
        subset=[bscore])
    # Drop features with too many NaNs
    sub_feats = sub[network_feat_cols].dropna(axis=1, thresh=int(0.8 * len(sub)))
    sub_feats = sub_feats.fillna(sub_feats.median())
    y_vec = sub[bscore].values
    # Need at least 20 subjects per score for PLS to be meaningful
    if len(sub) < 20 or sub_feats.shape[1] < 2:
        print(f"  Skipping {bscore}: n={len(sub)}")
        continue

    X_pls = StandardScaler().fit_transform(sub_feats.values)
    y_pls = (y_vec - y_vec.mean()) / y_vec.std()

    n_comp = min(3, X_pls.shape[1], X_pls.shape[0] - 1)
    pls = PLSRegression(n_components=n_comp, scale=False)
    pls.fit(X_pls, y_pls)

    # Cross-validated R² for each number of components
    from sklearn.model_selection import cross_val_score
    for nc in range(1, n_comp + 1):
        pls_cv = PLSRegression(n_components=nc, scale=False)
        scores = cross_val_score(pls_cv, X_pls, y_pls,
                                 cv=5, scoring='r2')
        pls_results.append({
            'Behavioral_Score': bscore,
            'n_components': nc,
            'n_subjects': len(sub),
            'CV_R2_mean': scores.mean(),
            'CV_R2_std':  scores.std(),
        })

    # Top loading features for component 1
    loadings = pd.Series(pls.x_loadings_[:, 0],
                         index=sub_feats.columns).abs().sort_values(
                             ascending=False)
    top_loadings = loadings.head(10)
    print(f"\n  PLS {bscore} (n={len(sub)}, 1-component CV R²="
          f"{pls_results[-n_comp]['CV_R2_mean']:.3f}):")
    print(f"  Top features: {top_loadings.index.tolist()[:5]}")

    # Save loading plot
    fig, ax = plt.subplots(figsize=(8, 4))
    top_loadings.plot(kind='barh', ax=ax)
    ax.set_title(f'PLS Component 1 Loadings — {bscore} (no-GSR)')
    ax.set_xlabel('|Loading|')
    plt.tight_layout()
    safe_name = bscore.replace('/', '_').replace(' ', '_')
    plt.savefig(f'/kaggle/working/plots/PLS_loadings_{safe_name}_GSR.pdf',
                bbox_inches='tight')
    plt.close()

if pls_results:
    pd.DataFrame(pls_results).to_csv(
        '/kaggle/working/results/PLS_results_GSR.csv', index=False)
    print(f"\nPLS results saved: {len(pls_results)} rows")

# ── (c) Hamiltonian vs other network metrics (exploratory) ───────────────
print("\n── Hamiltonian vs network metrics (within ASD and TD) ──")

hamil_corr_rows = []
for grp_name, grp_df in [('ASD', asd), ('TD', ctrl)]:
    energy_vals = grp_df['Whole_Energy_global'].dropna().values
    for feat in network_feat_cols:
        if feat == 'Whole_Energy_global':
            continue
        feat_vals = grp_df[feat].dropna()
        # Align on index
        common_idx = grp_df[['Whole_Energy_global', feat]].dropna().index
        if len(common_idx) < 10:
            continue
        x = grp_df.loc[common_idx, 'Whole_Energy_global'].values
        y = grp_df.loc[common_idx, feat].values
        pr, pp = stats.pearsonr(x, y)
        sr, sp = stats.spearmanr(x, y)
        hamil_corr_rows.append({
            'Group': grp_name, 'Network_Metric': feat,
            'Pearson_r': pr, 'Pearson_p': pp,
            'Spearman_rho': sr, 'Spearman_p': sp,
        })

hamil_corr_df = pd.DataFrame(hamil_corr_rows)
if not hamil_corr_df.empty:
    # FDR per group
    for grp_name in ['ASD', 'TD']:
        mask = hamil_corr_df['Group'] == grp_name
        if mask.sum() > 1:
            _, p_fdr, _, _ = multipletests(
                hamil_corr_df.loc[mask, 'Pearson_p'], method='fdr_bh')
            hamil_corr_df.loc[mask, 'Pearson_p_FDR'] = p_fdr
    hamil_corr_df = hamil_corr_df.sort_values(['Group', 'Pearson_p'])
    hamil_corr_df.to_csv(
        '/kaggle/working/results/hamiltonian_vs_metrics_GSR.csv',
        index=False)
    print(f"Hamiltonian correlations saved: {len(hamil_corr_df)} pairs")
    # Show top 10 per group
    for grp_name in ['ASD', 'TD']:
        top = (hamil_corr_df[hamil_corr_df['Group'] == grp_name]
               .head(10)[['Network_Metric', 'Pearson_r', 'Pearson_p']])
        print(f"\n  Top 10 correlates of H in {grp_name}:")
        print(top.to_string(index=False))

    # Heatmap of Hamiltonian vs intra/inter metrics
    for grp_name in ['ASD', 'TD']:
        sub_df = hamil_corr_df[hamil_corr_df['Group'] == grp_name].copy()
        sub_df['abs_r'] = sub_df['Pearson_r'].abs()
        top20 = sub_df.nlargest(20, 'abs_r')
        fig, ax = plt.subplots(figsize=(9, 5))
        colors = ['#d62728' if r > 0 else '#1f77b4'
                  for r in top20['Pearson_r']]
        ax.barh(top20['Network_Metric'], top20['Pearson_r'], color=colors)
        ax.axvline(0, color='black', lw=0.8)
        ax.set_title(f'Hamiltonian vs network metrics ({grp_name}, no-GSR)')
        ax.set_xlabel("Pearson r with Whole_Energy_global")
        plt.tight_layout()
        plt.savefig(
            f'/kaggle/working/plots/Hamiltonian_correlates_{grp_name}_GSR.pdf',
            bbox_inches='tight')
        plt.close()

for cat in ['Whole', 'Intra', 'Inter']:
    if stats_results.empty: 
        continue
    cat_results = stats_results[stats_results['Category'] == cat].copy()
    if len(cat_results) == 0: 
        continue
    
    sig = cat_results[cat_results['Significant_Uncorrected']].head(6)
    if len(sig) == 0:
        sig = cat_results.head(6)
    if len(sig) == 0: 
        continue
    
    n = len(sig)
    cols_n = min(3, n)
    rows_n = int(np.ceil(n / cols_n))
    fig, axes = plt.subplots(rows_n, cols_n, figsize=(5*cols_n, 4*rows_n))
    if n == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    
    for idx, (_, row) in enumerate(sig.iterrows()):
        feat = row['Feature']
        plot_data = harmonized_df[['Group', feat]].dropna()
        
        ax = axes[idx]
        for i, grp in enumerate(['Control', 'ASD']):
            grp_data = plot_data[plot_data['Group'] == grp][feat]
            color = '#00473a' if grp == 'Control' else '#c30061'
            bp = ax.boxplot([grp_data.values], positions=[i], widths=0.6, patch_artist=True)
            bp['boxes'][0].set_facecolor(color)
            bp['boxes'][0].set_alpha(0.7)
            ax.scatter([i]*len(grp_data), grp_data.values, c='black', alpha=0.4, s=15, zorder=3)
        
        ax.set_xticks([0, 1])
        ax.set_xticklabels(['Control', 'ASD'])
        title = feat.replace('Whole_', '').replace('Intra_', '').replace('Inter_', '').replace('__', ' ').replace('_', ' ')
        ax.set_title(f'{title}\np={row["p_value"]:.2e}, p_FDR={row["p_corrected"]:.2e}', fontsize=9)
    
    for idx in range(n, len(axes)): 
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/plots/{cat}_significant_GSR.pdf', dpi=150, bbox_inches='tight')
    plt.savefig(f'/kaggle/working/plots/{cat}_significant_GSR.png', dpi=150, bbox_inches='tight')
    plt.close()
    print(f"Saved {cat} plot")

if stats_results.empty:
    print("\nNo stats, skipping ML")
else:
    X = harmonized_df[fcols].copy()
    X = X.fillna(X.median())
    le = LabelEncoder()
    y = le.fit_transform(harmonized_df['Group'])
    
    print(f"\nML: X={X.shape}, y={len(y)}")
    
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
    print(f"Train={len(y_train)}, Test={len(y_test)}")
    
    rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    importance_df = pd.DataFrame({'Feature': X_train.columns, 'Importance': rf.feature_importances_}).sort_values('Importance', ascending=False)
    importance_df.to_csv('/kaggle/working/results/feature_importance_GSR.csv', index=False)
    
    n_sel = max(10, int(len(fcols) * 0.25))
    sel_feats = importance_df.head(n_sel)['Feature'].tolist()
    X_train_sel, X_test_sel = X_train[sel_feats].values, X_test[sel_feats].values
    
    models = {
        'GaussianNB': (Pipeline([('scaler', StandardScaler()), ('clf', GaussianNB())]), {'clf__var_smoothing': [1e-9, 1e-8, 1e-7]}),
        'SVM': (Pipeline([('scaler', StandardScaler()), ('clf', SVC(probability=True, class_weight='balanced', random_state=42))]), {'clf__C': [0.1, 1, 10], 'clf__gamma': ['scale', 0.001]}),
        'LogisticRegression': (Pipeline([('scaler', StandardScaler()), ('clf', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))]), {'clf__C': [0.01, 0.1, 1, 10]}),
        'KNN': (Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier())]), {'clf__n_neighbors': [5, 9, 15]}),
        'XGBoost': (Pipeline([('scaler', StandardScaler()), ('clf', XGBClassifier(eval_metric='logloss', random_state=42, verbosity=0))]), {'clf__n_estimators': [100], 'clf__max_depth': [3, 5], 'clf__learning_rate': [0.01, 0.1]})
    }
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    ml_results = []
    best_model_info = None
    best_acc = 0
    
    for name, (pipe, params) in models.items():
        grid = GridSearchCV(pipe, params, cv=cv, scoring='accuracy', n_jobs=-1)
        grid.fit(X_train_sel, y_train)
        y_pred = grid.predict(X_test_sel)
        y_proba = grid.predict_proba(X_test_sel)[:, 1]
        acc = accuracy_score(y_test, y_pred)
        bal_acc = balanced_accuracy_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred, average='macro')
        ml_results.append({'model': name, 'best_params': str(grid.best_params_), 'cv_score': grid.best_score_, 'test_acc': acc, 'test_bal_acc': bal_acc, 'test_f1_macro': f1})
        print(f"{name}: CV={grid.best_score_:.3f}, Test={acc:.3f}")
        if acc > best_acc:
            best_acc = acc
            best_model_info = (name, grid.best_estimator_, y_pred, y_proba, y_test)
    
    ml_results_df = pd.DataFrame(ml_results).sort_values('test_acc', ascending=False)
    ml_results_df.to_csv('/kaggle/working/results/ml_results_GSR.csv', index=False)
    
    if best_model_info:
        name, model, y_pred, y_proba, y_true = best_model_info
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        
        cm = confusion_matrix(y_true, y_pred)
        im = axes[0].imshow(cm, cmap='Blues')
        axes[0].set_xticks([0, 1])
        axes[0].set_yticks([0, 1])
        axes[0].set_xticklabels(['ASD', 'Control'])
        axes[0].set_yticklabels(['ASD', 'Control'])
        axes[0].set_xlabel('Predicted')
        axes[0].set_ylabel('True')
        axes[0].set_title(f'Confusion Matrix ({name})')
        for i in range(2):
            for j in range(2):
                axes[0].text(j, i, str(cm[i, j]), ha='center', va='center', fontsize=14)
        
        fpr, tpr, _ = roc_curve(y_true, y_proba)
        roc_auc = auc(fpr, tpr)
        axes[1].plot(fpr, tpr, 'crimson', lw=2, label=f'AUC = {roc_auc:.2f}')
        axes[1].plot([0,1], [0,1], 'gray', lw=2, linestyle='--')
        axes[1].set_xlabel('False Positive Rate')
        axes[1].set_ylabel('True Positive Rate')
        axes[1].set_title(f'ROC Curve ({name})')
        axes[1].legend(loc='lower right')
        
        plt.tight_layout()
        plt.savefig('/kaggle/working/plots/best_model_GSR.pdf', dpi=150, bbox_inches='tight')
        plt.savefig('/kaggle/working/plots/best_model_GSR.png', dpi=150, bbox_inches='tight')
        plt.close()
        
        pd.DataFrame({'y_true': y_true, 'y_pred': y_pred, 'y_proba': y_proba}).to_csv('/kaggle/working/results/best_model_predictions_GSR.csv', index=False)
        print(f"Saved best model results")
    
    plt.figure(figsize=(10, 6))
    df_plot = ml_results_df.sort_values('test_acc', ascending=False)
    bars = plt.bar(df_plot['model'], df_plot['test_acc'], color='steelblue', edgecolor='black')
    for bar, acc in zip(bars, df_plot['test_acc']):
        plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01, f'{acc:.3f}', ha='center')
    plt.xlabel('Model')
    plt.ylabel('Test Accuracy')
    plt.title('Model Comparison (No GSR)')
    plt.ylim(0, 1)
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.savefig('/kaggle/working/plots/model_comparison_GSR.pdf', dpi=150, bbox_inches='tight')
    plt.savefig('/kaggle/working/plots/model_comparison_GSR.png', dpi=150, bbox_inches='tight')
    plt.close()

import shutil
os.makedirs('/kaggle/working/all_GSR', exist_ok=True)
# (already in your code — just make sure these new files copy correctly)
for f in os.listdir('/kaggle/working/results'):
    shutil.copy(f'/kaggle/working/results/{f}', '/kaggle/working/all_GSR/')
for f in os.listdir('/kaggle/working/plots'):
    shutil.copy(f'/kaggle/working/plots/{f}', '/kaggle/working/all_GSR/')
shutil.make_archive('/kaggle/working/GSR_results', 'zip',
                    '/kaggle/working/all_GSR')

print("\n" + "="*60)
print("SUMMARY")
print("="*60)
print(f"Subjects: {len(harmonized_df)} (ASD: {sum(harmonized_df['Group']=='ASD')}, Control: {sum(harmonized_df['Group']=='Control')})")
if not stats_results.empty:
    for cat in ['Whole', 'Intra', 'Inter']:
        cr = stats_results[stats_results['Category'] == cat]
        if len(cr) > 0:
            print(f"{cat}: {cr['Significant_FDR'].sum()} FDR-sig, {cr['Significant_Uncorrected'].sum()} uncorrected")
    if 'ml_results_df' in dir():
        print(f"Best model: {ml_results_df.iloc[0]['model']} ({ml_results_df.iloc[0]['test_acc']:.3f})")
        if best_model_info:
            fpr, tpr, _ = roc_curve(best_model_info[4], best_model_info[3])
            print(f"AUC: {auc(fpr, tpr):.3f}")
print("\nDownload: /kaggle/working/GSR_results.zip")

  Preparing metadata (setup.py) ... done
Found 182 ROI files
Extracted features: (182, 96)
ASD: 90, Control: 92
After outlier removal: 167
ASD: 80, Control: 87
Saved raw features
Feature columns to harmonize: 92
Sites: 13
Site mapping: {'USM': 0, 'MAX_MUN': 1, 'LEUVEN_1': 2, 'OLIN': 3, 'NYU': 4, 'TRINITY': 5, 'SBL': 6, 'UM_1': 7, 'PITT': 8, 'CALTECH': 9, 'UCLA_1': 10, 'UM_2': 11, 'CMU': 12}
Data matrix shape (subjects x features): (167, 92)
NaN count before filling: 0
NaN count after filling: 0
Data range: [-345.597, 35.528]
Data for ComBat shape (features x subjects): (92, 167)

Running ComBat harmonization...
[neuroCombat] Creating design matrix
[neuroCombat] Standardizing data across features
[neuroCombat] Fitting L/S model and finding priors
[neuroCombat] Finding parametric adjustments
[neuroCombat] Final adjustment of data
ComBat output shape: (92, 167)
ComBat output NaN count: 15364
ComBat output range: [nan, nan]
Saved harmonized features

Verification:
  Shape: (167, 97)
  Whol

# Reviewer-revision add-on analyses (no-GSR pipeline)

**How to use:** run your existing `no-gsr-analysis` notebook first, then run the cells
below **in the same kernel**. They reuse the objects your notebook already created:
`harmonized_df`, `asd`, `ctrl`, `asd_merged`, `network_feat_cols`, `behav_score_cols`,
`pheno_df`, `roi_files`, `file_to_sub`, `sub_to_dx`, `sub_to_site`, `sub_to_iq`,
`load_ts`, `compute_fc`, `extract_file_id`, `adaptive_test_full`.

Each section addresses one reviewer comment. Run what you need; report the printed
numbers (and the saved CSVs) back. The cells are written defensively but were not
executed against the live data, so if anything errors, paste the traceback back.


## (2b) Partial Least Squares Correlation (PLSC) — behavioural PLS

The reviewer suggested PLSC as the descriptive multivariate analogue of the bivariate
correlations. PLSC does an SVD of the brain x behaviour cross-correlation matrix and is
assessed by **permutation testing** (significance of each latent variable) and
**bootstrap ratios** (stability of brain saliences). No prediction / cross-validation.


In [2]:

import numpy as np, pandas as pd
from numpy.linalg import svd

def run_plsc(Xdf, Ydf, n_perm=5000, n_boot=1000, seed=0):
    """PLSC (behavioural PLS). Xdf: subjects x brain features; Ydf: subjects x behaviours.
    Returns singular values, % covariance, per-LV permutation p, LV1 saliences and
    bootstrap ratios (|BSR|>~2 ~ stable)."""
    rng = np.random.default_rng(seed)
    X = Xdf.values.astype(float); Y = Ydf.values.astype(float)
    Xz = (X - X.mean(0)) / (X.std(0, ddof=1) + 1e-12)
    Yz = (Y - Y.mean(0)) / (Y.std(0, ddof=1) + 1e-12)
    n = Xz.shape[0]
    R = Yz.T @ Xz / (n - 1)                      # behaviour x brain
    U, S, Vt = svd(R, full_matrices=False)       # U: behaviour saliences; Vt: brain saliences
    cov_exp = S**2 / np.sum(S**2)
    # permutation test on singular values (permute subject order of Y)
    perm_ge = np.zeros(len(S))
    for _ in range(n_perm):
        idx = rng.permutation(n)
        sp = svd(Yz[idx].T @ Xz / (n - 1), compute_uv=False)
        m = min(len(sp), len(S))
        perm_ge[:m] += (sp[:m] >= S[:m])
    p_vals = (perm_ge + 1) / (n_perm + 1)
    # bootstrap ratios for LV1 brain saliences
    boot = np.zeros((n_boot, Vt.shape[1]))
    for bi in range(n_boot):
        b = rng.integers(0, n, n)
        Xb = (X[b]-X[b].mean(0))/(X[b].std(0,ddof=1)+1e-12)
        Yb = (Y[b]-Y[b].mean(0))/(Y[b].std(0,ddof=1)+1e-12)
        _, _, Vtb = svd(Yb.T @ Xb/(n-1), full_matrices=False)
        if np.dot(Vtb[0], Vt[0]) < 0: Vtb[0] = -Vtb[0]
        boot[bi] = Vtb[0]
    bsr = Vt[0] / (boot.std(0) + 1e-12)
    return dict(S=S, cov_exp=cov_exp, p_vals=p_vals, n=n,
                brain_cols=list(Xdf.columns), behav_cols=list(Ydf.columns),
                behav_salience_LV1=U[:,0], brain_salience_LV1=Vt[0], brain_bsr_LV1=bsr)

def plsc_report(name, behav_subset):
    cols = [c for c in behav_subset if c in asd_merged.columns]
    block = asd_merged[['Subject'] + network_feat_cols + cols].copy()
    block = block.dropna(subset=cols)                       # complete cases on behaviour
    feats = [c for c in network_feat_cols if block[c].notna().mean() > 0.8]
    block[feats] = block[feats].fillna(block[feats].median())
    block = block.dropna(subset=feats)
    if len(block) < 20 or len(feats) < 2:
        print(f"[{name}] insufficient data (n={len(block)})"); return None
    res = run_plsc(block[feats], block[cols])
    print(f"\n=== PLSC: {name}  (n={res['n']}, behaviours={cols}) ===")
    for k,(sv,ce,pv) in enumerate(zip(res['S'], res['cov_exp'], res['p_vals'])):
        print(f"  LV{k+1}: singular value={sv:.3f}, cov explained={ce*100:5.1f}%, perm p={pv:.4f}")
    print("  LV1 behaviour saliences:",
          {c: round(float(s),3) for c,s in zip(res['behav_cols'], res['behav_salience_LV1'])})
    bsr = pd.Series(res['brain_bsr_LV1'], index=res['brain_cols']).sort_values(key=np.abs, ascending=False)
    print("  LV1 top brain saliences by |bootstrap ratio| (|BSR|>2 ~ stable):")
    print(bsr.head(12).round(2).to_string())
    out = pd.DataFrame({'feature':res['brain_cols'],
                        'LV1_salience':res['brain_salience_LV1'],
                        'LV1_bootstrap_ratio':res['brain_bsr_LV1']}).sort_values(
                        'LV1_bootstrap_ratio', key=np.abs, ascending=False)
    safe = name.replace(' ','_').replace('/','_')
    out.to_csv(f'/kaggle/working/results/PLSC_{safe}_GSR.csv', index=False)
    return res

# ADOS block (larger n) and full block
_ = plsc_report('ADOS_block', ['ADOS Total','ADOS Communication'])
_ = plsc_report('Full_block', behav_score_cols)
print("\nPLSC CSVs saved in /kaggle/working/results/")




=== PLSC: ADOS_block  (n=52, behaviours=['ADOS Total', 'ADOS Communication']) ===
  LV1: singular value=1.491, cov explained= 81.5%, perm p=0.8784
  LV2: singular value=0.710, cov explained= 18.5%, perm p=0.4957
  LV1 behaviour saliences: {'ADOS Total': -0.693, 'ADOS Communication': -0.721}
  LV1 top brain saliences by |bootstrap ratio| (|BSR|>2 ~ stable):
Intra_SomatoMotor_AvgWeight                   -4.99
Intra_Salience/VentAttn_AvgWeight              3.09
Inter_Visual__Limbic_AvgWeight                 2.60
Inter_Visual__Salience/VentAttn_EnergyBlock   -2.44
Inter_Visual__Salience/VentAttn_AvgFalffW      2.44
Inter_Limbic__Default_EnergyBlock             -2.41
Inter_Limbic__Default_AvgFalffW                2.41
Intra_Frontoparietal_AvgFalffW                -2.38
Intra_Frontoparietal_EnergyBlock               2.38
Inter_Visual__Salience/VentAttn_AvgWeight     -2.23
Intra_Limbic_AvgFalffW                        -2.11
Intra_Limbic_EnergyBlock                       2.11

=== PLSC: Full_

## (2c) Signed modularity (Rubinov & Sporns, 2011)

A weight-conserving, **sign-aware** measure, addressing the concern that absolute-valued
graph efficiency mixes positive and negative connection systems. This recomputes the FC
matrices (your pipeline discards them) and computes signed modularity Q on the signed FC.

It reports (i) the ASD-vs-TD group comparison, (ii) the within-group correlation of Q with
the Hamiltonian, and (iii) Q vs the existing efficiency features — mirroring how efficiency
is reported.

**Important trade-off if you fold Q into the manuscript's Whole-brain FDR family:** that
family currently has 8 features (bipolarity's no-GSR FDR p = 0.063 with 8 tests). Adding Q
makes it 9 and will shift bipolarity's FDR p upward. The cell below therefore reports Q as a
*complementary* measure (its own test + correlations), not inside the bipolarity FDR family.
Decide which framing you want and I will write the matching text.


In [3]:

import numpy as np, pandas as pd
from scipy import stats
try:
    import bct
except Exception:
    import sys; print("installing bctpy..."); 
    get_ipython().system('pip install bctpy -q')
    import bct

def signed_modularity(fc, gamma=1.0, reps=10, seed=0):
    W = np.nan_to_num(np.array(fc, dtype=float)); np.fill_diagonal(W, 0)
    best = -np.inf
    for r in range(reps):
        try:
            _, q = bct.community_louvain(W, gamma=gamma, B='negative_sym', seed=seed+r)
            best = max(best, q)
        except Exception:
            pass
    return best if np.isfinite(best) else np.nan

# recompute FC per subject and compute signed modularity
mod_rows = []
for fpath in roi_files:
    fid = extract_file_id(fpath); sub = file_to_sub.get(fid)
    if sub is None: continue
    try:
        ts = load_ts(fpath)
        if ts.ndim == 1 or ts.shape[1] != 200 or ts.shape[0] < 50: continue
        q = signed_modularity(compute_fc(ts))
        mod_rows.append({'Subject': sub, 'SignedModularity': q})
    except Exception:
        pass
mod_df = pd.DataFrame(mod_rows).dropna()
print(f"Signed modularity computed for {len(mod_df)} subjects")

# restrict to the analysed (harmonized) cohort and attach Group + H + efficiency
keep = harmonized_df[['Subject','Group','Whole_Energy_global',
                      'Whole_GlobalEfficiency','Whole_LocalEfficiency']].copy()
m = keep.merge(mod_df, on='Subject', how='inner')
print(f"Matched to analysed cohort: {len(m)} "
      f"(ASD={sum(m['Group']=='ASD')}, TD={sum(m['Group']=='Control')})")

a = m[m['Group']=='ASD']['SignedModularity'].dropna()
t = m[m['Group']=='Control']['SignedModularity'].dropna()
res = adaptive_test_full(a.values, t.values)
print(f"\nGroup comparison (note: not ComBat-harmonised here):")
print(f"  ASD  mean={a.mean():.4f} (SD {a.std():.4f})")
print(f"  TD   mean={t.mean():.4f} (SD {t.std():.4f})")
print(f"  test={res[0]}, p={res[2]:.4f}")

for g,lbl in [('ASD','ASD'),('Control','TD')]:
    sub = m[m['Group']==g].dropna(subset=['SignedModularity','Whole_Energy_global'])
    if len(sub) > 10:
        r,p = stats.pearsonr(sub['SignedModularity'], sub['Whole_Energy_global'])
        rg,pg = stats.pearsonr(sub['SignedModularity'], sub['Whole_GlobalEfficiency'])
        print(f"\n[{lbl}] Q vs H:  r={r:.3f}, p={p:.4f}  (n={len(sub)})")
        print(f"[{lbl}] Q vs global efficiency:  r={rg:.3f}, p={pg:.4f}")
m.to_csv('/kaggle/working/results/signed_modularity_GSR.csv', index=False)
print("\nSaved signed_modularity_GSR.csv")



installing bctpy...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.4/110.4 kB 2.4 MB/s eta 0:00:00
Signed modularity computed for 182 subjects
Matched to analysed cohort: 167 (ASD=80, TD=87)

Group comparison (note: not ComBat-harmonised here):
  ASD  mean=0.3115 (SD 0.0295)
  TD   mean=0.3122 (SD 0.0246)
  test=Mann-Whitney U, p=0.6480

[ASD] Q vs H:  r=0.043, p=0.7076  (n=80)
[ASD] Q vs global efficiency:  r=0.518, p=0.0000

[TD] Q vs H:  r=0.034, p=0.7570  (n=87)
[TD] Q vs global efficiency:  r=0.481, p=0.0000

Saved signed_modularity_GSR.csv


## (3b) Same-sample comparison (no-GSR restricted to the GSR sample)

The reviewer asked that both pipelines use the same participants. Point this at the
**Subject-ID list from your GSR analysis** (the ComBat features CSV the GSR notebook saved).
It restricts the no-GSR whole-brain comparisons to the intersection and reprints them, so
you can state the no-GSR results on the identical GSR sample.


In [4]:

import numpy as np, pandas as pd
from scipy import stats
from statsmodels.stats.multitest import multipletests

GSR_FEATURES_CSV = ''   # <-- set to e.g. '/kaggle/input/your-gsr-run/features_ComBat_GSR.csv'

whole_feats = [c for c in harmonized_df.columns if c.startswith('Whole_')]
if not GSR_FEATURES_CSV:
    print("Set GSR_FEATURES_CSV to your GSR features CSV path, then re-run this cell.")
else:
    gsr_df = pd.read_csv(GSR_FEATURES_CSV)
    gsr_subjects = set(gsr_df['Subject'].astype(str))
    common = harmonized_df[harmonized_df['Subject'].astype(str).isin(gsr_subjects)].copy()
    print(f"Common subjects (no-GSR ∩ GSR): {len(common)} "
          f"(ASD={sum(common['Group']=='ASD')}, TD={sum(common['Group']=='Control')})")
    a_df = common[common['Group']=='ASD']; t_df = common[common['Group']=='Control']
    rows = []
    for f in whole_feats:
        g1 = a_df[f].dropna().values; g2 = t_df[f].dropna().values
        if len(g1) < 3 or len(g2) < 3: continue
        r = adaptive_test_full(g1, g2)
        rows.append({'Feature': f, 'ASD_mean': g1.mean(), 'TD_mean': g2.mean(),
                     'Test': r[0], 'p': r[2]})
    out = pd.DataFrame(rows)
    if len(out):
        out['p_FDR'] = multipletests(out['p'], method='fdr_bh')[1]
        out = out.sort_values('p')
        print("\nWhole-brain comparisons on the SAME (GSR) sample, no-GSR data:")
        print(out.to_string(index=False))
        out.to_csv('/kaggle/working/results/GSR_on_GSRsample_whole.csv', index=False)
        print("\nSaved GSR_on_GSRsample_whole.csv")



Set GSR_FEATURES_CSV to your GSR features CSV path, then re-run this cell.


## Reporting back

Please send: the printed PLSC LV table (singular values, % covariance, permutation p) and
LV1 saliences; the signed-modularity group means/p and its Q–H correlations; and, if you set
the GSR CSV, the same-sample whole-brain table. I will fold the numbers into the manuscript
and write the matching Methods/Results sentences (and add a supplementary PLSC figure).
